In [ ]:
# 1. IMPORTAR BIBLIOTECAS
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# 2. CARGAR EL DATASET
print("CARGANDO DATASET TITANIC...")

# Si el archivo no existe localmente, lo descargamos automáticamente de un repositorio público
if not os.path.exists("train.csv"):
    print("Archivo 'train.csv' no encontrado localmente. Descargándolo de internet...")
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    datos = pd.read_csv(url)
    datos.to_csv("train.csv", index=False)
else:
    datos = pd.read_csv("train.csv")

print("\nPrimeros registros:")
print(datos.head())

print("\nInformacion del dataset:")
print(datos.info())

print("\nValores faltantes:")
print(datos.isnull().sum())


# 3. SELECCIONAR CARACTERISTICAS
# Variables que utilizara el modelo
caracteristicas = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked"
]

X = datos[caracteristicas]

# Variable que queremos predecir
y = datos["Survived"]


# 4. PREPARAR LOS DATOS
# Variables numericas
numericas = [
    "Pclass", "Age", "SibSp",
    "Parch", "Fare"
]

# Variables categoricas
categoricas = ["Sex", "Embarked"]

# Completar datos numericos faltantes
procesamiento_numerico = SimpleImputer(
    strategy="median"
)

# Completar datos categoricos faltantes
procesamiento_categorico = Pipeline([
    ("imputar", SimpleImputer(strategy="most_frequent")),
    ("codificar", OneHotEncoder(handle_unknown="ignore"))
])

# Combinar los procesamientos
preparador = ColumnTransformer([
    ("numerico", procesamiento_numerico, numericas),
    ("categorico", procesamiento_categorico, categoricas)
])

# 5. DIVIDIR LOS DATOS
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

# 6. CREAR EL MODELO
modelo = Pipeline([
    ("preparacion", preparador),

    ("clasificador", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])


# 7. ENTRENAR EL MODELO
print("\nENTRENANDO MODELO...")

modelo.fit(X_entrenamiento, y_entrenamiento)

print("ENTRENAMIENTO COMPLETADO")


# 8. EVALUAR EL MODELO
predicciones = modelo.predict(X_prueba)

exactitud = accuracy_score(
    y_prueba,
    predicciones
)

print("\nRESULTADOS DEL MODELO")

print("Exactitud:", round(exactitud * 100, 2), "%")

print("\nReporte de clasificacion:")
print(classification_report(
    y_prueba,
    predicciones
))


# 9. PREDICCION INTERACTIVA
def predecir_pasajero():

    print("\n================================")
    print(" PREDICCION DE SUPERVIVENCIA")
    print("================================")

    try:
        clase = int(input("Clase del boleto (1, 2 o 3): "))
        sexo = input("Sexo (W = Mujer / M = Hombre): ").upper()
        edad = float(input("Edad del pasajero: "))
        hermanos = int(input("Hermanos o compañero a bordo: "))
        padres = int(input("Padres o hijos a bordo: "))
        tarifa = float(input("Precio del boleto: "))
        puerto = input("Puerto (S, C o Q): ").upper()

        # Validar datos
        if clase not in [1, 2, 3]:
            print("Clase no valida.")
            return

        if sexo == "W":
            sexo = "female"

        elif sexo == "M":
            sexo = "male"

        else:
            print("Sexo no valido. Utiliza W o M.")
            return

        if edad < 0 or tarifa < 0 or hermanos < 0 or padres < 0:
            print("Los valores no pueden ser negativos.")
            return

        if puerto not in ["S", "C", "Q"]:
            print("Puerto no valido.")
            return

        # Crear dataframe del nuevo pasajero
        nuevo_pasajero = pd.DataFrame([{
            "Pclass": clase,
            "Sex": sexo,
            "Age": edad,
            "SibSp": hermanos,
            "Parch": padres,
            "Fare": tarifa,
            "Embarked": puerto
        }])

        # Realizar prediccion
        resultado = modelo.predict(nuevo_pasajero)

        probabilidades = modelo.predict_proba(
            nuevo_pasajero
        )

        probabilidad = probabilidades[0][1] * 100

        print("\nRESULTADO DE LA PREDICCION")

        if resultado[0] == 1:
            print("El modelo predice que SOBREVIVIRIA.")
        else:
            print("El modelo predice que NO SOBREVIVIRIA.")

        print(
            "Probabilidad estimada de supervivencia:",
            round(probabilidad, 2), "%"
        )

    except ValueError:
        print("Error: introduce valores numericos validos.")


# 10. MENU PRINCIPAL
while True:

    print("\n========== MENU TITANIC ==========")
    print("1. Predecir supervivencia")
    print("2. Mostrar informacion del dataset")
    print("3. Salir")

    opcion = input("Selecciona una opcion: ")

    if opcion == "1":
        predecir_pasajero()

    elif opcion == "2":
        print(datos.head(10))

        print("\nTotal de pasajeros:", len(datos))

        print("\nSupervivientes:")
        print(datos["Survived"].value_counts())

    elif opcion == "3":
        print("Finalizando programa...")
        break

    else:
        print("Opcion incorrecta.")

CARGANDO DATASET TITANIC...

Primeros registros:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4     